In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")


In [ ]:
# Local / SSH: Google Drive mounting is not needed.
# Keep the image_realness_project folder on the SSH server and open it in VS Code.


In [ ]:
# Local / SSH setup for VS Code notebooks
# Run this from anywhere inside the image_realness_project folder.
import sys
from pathlib import Path
import torch


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "core").is_dir() and (candidate / "scripts").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find image_realness_project root. "
        "Open VS Code on the project folder or start the notebook from inside it."
    )


PROJECT = find_project_root()
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PROJECT:", PROJECT)
print("device:", device)


In [ ]:
from core.models.joint_model import JointModel

joint_model = JointModel(use_pretrained_backbones=True).to(device)
state = torch.load(PROJECT / "checkpoints" / "JOINT_2024.pth", map_location=device)
joint_model.load_state_dict(state)
joint_model.eval()
print("JOINT loaded")

In [ ]:
from core.guidance.rationality_guidance import RationalityGuidanceScorer

scorer = RationalityGuidanceScorer(joint_model, image_size=224, blur_kernel=11).to(device)
scorer.eval()

In [ ]:
from core.generation.sd_generator import load_sd_pipeline

pipe = load_sd_pipeline(project=PROJECT, device=device.type)
pipe.safety_checker = None   # skip for speed
print("SD loaded")

In [ ]:
# tweak these
PROMPT              = "a golden retriever in a sunlit meadow"
NUM_INFERENCE_STEPS = 30
GUIDANCE_SCALE      = 7.5
RATIONALITY_WEIGHT  = 0.05   # increase if effect too weak, decrease if images degrade
GUIDANCE_LAST_STEPS = 20        # only guide the last N denoising steps
SEED                = 42
OUTPUT_DIR          = PROJECT / "outputs" / "images" / "rationality_guided"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# run — this is the generate_with_rationality_guidance script as a function call
import subprocess, sys

result = subprocess.run([
    sys.executable,
    str(PROJECT / "scripts" / "generate_with_rationality_guidance.py"),
    "--project",              str(PROJECT),
    "--prompt",               PROMPT,
    "--output-dir",           str(OUTPUT_DIR),
    "--num-inference-steps",  str(NUM_INFERENCE_STEPS),
    "--guidance-scale",       str(GUIDANCE_SCALE),
    "--rationality-weight",   str(RATIONALITY_WEIGHT),
    "--guidance-last-steps",  str(GUIDANCE_LAST_STEPS),
    "--seed",                 str(SEED),
], capture_output=False)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

images = sorted(OUTPUT_DIR.glob("guided_*.png"))
fig, axes = plt.subplots(1, len(images), figsize=(5 * len(images), 5))
if len(images) == 1:
    axes = [axes]
for ax, p in zip(axes, images):
    ax.imshow(Image.open(p))
    ax.set_title(p.name)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
from core.generation.sd_generator import load_sd_pipeline, generate_image
from pathlib import Path
import torch
import matplotlib.pyplot as plt
from PIL import Image

PROMPT = "a person sitting in park"
SEED   = 42  # same seed so the only difference is guidance

# --- without guidance (standard SD) ---
no_guidance_path = PROJECT / "outputs" / "images" / "no_guidance" / "image_000.png"
generate_image(
    pipe=pipe,
    prompt=PROMPT,
    output_path=no_guidance_path,
    guidance_scale=7.5,
    num_inference_steps=30,
    seed=SEED,
)

# --- with rationality guidance (run the script) ---
import subprocess, sys
guided_dir = PROJECT / "outputs" / "images" / "rationality_guided_compare"

subprocess.run([
    sys.executable,
    str(PROJECT / "scripts" / "generate_with_rationality_guidance.py"),
    "--project",             str(PROJECT),
    "--prompt",              PROMPT,
    "--output-dir",          str(guided_dir),
    "--num-inference-steps", "30",
    "--guidance-scale",      "7.5",
    "--rationality-weight",  "0.05",
    "--guidance-last-steps", "15",
    "--seed",                str(SEED),
])

# --- display side by side ---
img_no_guidance = Image.open(no_guidance_path)
img_guided      = Image.open(guided_dir / "guided_000.png")

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(img_no_guidance)
axes[0].set_title("Without Guidance (CFG only)", fontsize=13)
axes[0].axis("off")

axes[1].imshow(img_guided)
axes[1].set_title("With Rationality Guidance", fontsize=13)
axes[1].axis("off")

plt.suptitle(f'Prompt: "{PROMPT}"', fontsize=11, y=1.01)
plt.tight_layout()
plt.show()